# Uncovered Interest Rate Parity (UIP): USA vs. Euro Area
## Empirical Analysis Based on Quarterly Data 2022–2025

---

**Course:** Fundamentals of Exchange Rate Theory and Exchange Rate Markets  
**Level:** Bachelor Economics

---

## How to Use This Notebook

You do **not** need to write any code. Your task is to **understand and interpret** each step.

**Run a cell:** Click on the cell, then press **Shift + Enter**  
**Important:** Always run cells **from top to bottom**!

In [ ]:
required_packages <- c("dplyr","ggplot2","lubridate","tidyr","scales","knitr")
for (pkg in required_packages) {
  if (!require(pkg, character.only=TRUE, quietly=TRUE)) {
    install.packages(pkg, repos="https://cran.rstudio.com/", quiet=TRUE)
    library(pkg, character.only=TRUE, quietly=TRUE)
  }
}
options(repr.plot.width=10, repr.plot.height=5)

In [ ]:
us_raw <- read.csv("DTB3.csv"); colnames(us_raw) <- c("date","value")
eu_raw <- read.csv("IR3TIB01DEM156N.csv"); colnames(eu_raw) <- c("date","value")
fx_raw <- read.csv("DEXUSEU.csv"); colnames(fx_raw) <- c("date","value")
clean <- function(df) {
  df |> dplyr::mutate(date=as.Date(date),
        value=suppressWarnings(as.numeric(value))) |>
        dplyr::filter(!is.na(value))
}
us_raw <- clean(us_raw); eu_raw <- clean(eu_raw); fx_raw <- clean(fx_raw)

In [ ]:
start_date <- as.Date("2022-01-01"); end_date <- as.Date("2025-12-31")
to_quarterly <- function(df, varname) {
  df |> filter(date>=start_date, date<=end_date) |>
    mutate(month_start=floor_date(date,"month")) |>
    group_by(month_start) |>
    summarise(value=mean(value,na.rm=TRUE),.groups="drop") |>
    mutate(qstart=floor_date(month_start,"quarter")) |>
    filter(month_start==qstart) |> select(qstart,value) |>
    rename(!!varname:=value)
}
us_q <- to_quarterly(us_raw,"i_US")
eu_q <- to_quarterly(eu_raw,"i_EU")
fx_q <- to_quarterly(fx_raw,"S_start")
fx_end_raw <- fx_raw |>
  filter(date>=start_date, date<=end_date+months(3)) |>
  mutate(month_start=floor_date(date,"month")) |>
  group_by(month_start) |>
  summarise(value=mean(value,na.rm=TRUE),.groups="drop") |>
  mutate(monthnum=as.integer(format(month_start,"%m")),
         qstart=floor_date(month_start,"quarter")) |>
  filter(monthnum %in% c(1,4,7,10)) |>
  mutate(qstart=qstart-months(3)) |>
  filter(qstart>=start_date) |> select(qstart,value) |> rename(S_end=value)
df <- us_q |> inner_join(eu_q,by="qstart") |>
  inner_join(fx_q,by="qstart") |> inner_join(fx_end_raw,by="qstart") |>
  filter(!is.na(i_US),!is.na(i_EU),!is.na(S_start),!is.na(S_end)) |>
  mutate(
    zins_diff      = i_US - i_EU,
    S_UIP          = S_start * (1+i_US/400)/(1+i_EU/400),
    delta_s_actual = (S_end/S_start-1)*100,
    delta_s_uip    = (S_UIP/S_start-1)*100,
    abweichung     = delta_s_actual - delta_s_uip,
    abweichung_pct = (S_end/S_UIP-1)*100
  )

---

## 1. Interactive UIP Explorer

Before diving into the data, let's build an **intuition** for the UIP.

The UIP states that the expected exchange rate change over the **next three months** must exactly offset the interest rate differential. The sliders show **annualised** interest rates, but the prediction refers to the **quarterly change** (3 months), using:

$$S_1 = S_0 \cdot \frac{1 + i_{US}/400}{1 + i_{EU}/400}$$

**Use the sliders** to explore how the UIP-implied exchange rate responds to interest rate shifts.

In [ ]:
library(IRdisplay)
html_code <- '
<style>
.uip-box{font-family:Calibri,Arial,sans-serif;max-width:700px;background:#f8f9fa;
  border:1px solid #dee2e6;border-radius:8px;padding:24px;margin:10px 0}
.uip-title{font-size:17px;font-weight:bold;margin-bottom:6px;color:#212529}
.uip-sub{font-size:13px;color:#6c757d;margin-bottom:16px}
.slider-row{display:flex;align-items:center;margin:10px 0;gap:12px}
.slider-label{width:270px;font-size:14px;color:#495057}
input[type=range]{flex:1;accent-color:#1f77b4}
.slider-val{width:55px;text-align:right;font-weight:bold;font-size:14px;color:#1f77b4}
.result-box{margin-top:20px;background:white;border-left:4px solid #1f77b4;
  padding:14px 18px;border-radius:4px}
.result-row{display:flex;justify-content:space-between;margin:6px 0;font-size:14px}
.result-label{color:#495057}.result-val{font-weight:bold}
.pos{color:#d62728}.neg{color:#2ca02c}.neu{color:#333}
.interp{margin-top:14px;padding:10px 14px;border-radius:4px;font-size:14px;line-height:1.6}
.dep{background:#fff3cd;border:1px solid #ffc107}
.app{background:#d4edda;border:1px solid #28a745}
.par{background:#e2e3e5;border:1px solid #adb5bd}
.note{font-size:18px;font-family:Calibri,Arial,sans-serif;color:#6c757d;margin-top:12px;font-style:italic}
</style>
<div class="uip-box">
<div class="uip-title">UIP Interactive Explorer</div>
<div class="uip-sub">Anchor: Q1 2022 &mdash; S&#8320; = 1.1317 USD/EUR &nbsp;|&nbsp; Sliders: annualised rates &nbsp;|&nbsp; Prediction: 3-month horizon</div>
<div class="slider-row">
  <span class="slider-label">US Interest Rate i<sub>US</sub> (% p.a., annualised)</span>
  <input type="range" id="iUS" min="-2" max="8" step="0.25" value="0.15" oninput="calcUIP()">
  <span class="slider-val" id="vUS">0.15</span>
</div>
<div class="slider-row">
  <span class="slider-label">Euro Area Interest Rate i<sub>EU</sub> (% p.a., annualised)</span>
  <input type="range" id="iEU" min="-2" max="8" step="0.25" value="-0.56" oninput="calcUIP()">
  <span class="slider-val" id="vEU">-0.56</span>
</div>
<div class="result-box">
  <div class="result-row"><span class="result-label">Spot Rate S&#8320; (USD/EUR)</span><span class="result-val neu">1.1317</span></div>
  <div class="result-row"><span class="result-label">Interest Differential i<sub>US</sub> &minus; i<sub>EU</sub> (annualised)</span><span class="result-val" id="rDiff">--</span></div>
  <div class="result-row"><span class="result-label">Quarterly differential (= annual &divide; 4)</span><span class="result-val" id="rDiffQ">--</span></div>
  <div class="result-row"><span class="result-label">UIP-Implied Rate S&#8321; in 3 months (USD/EUR)</span><span class="result-val" id="rS1">--</span></div>
  <div class="result-row"><span class="result-label">Predicted USD Change over 3 months</span><span class="result-val" id="rChg">--</span></div>
</div>
<div class="interp par" id="interp">Adjust the sliders to see the UIP prediction.</div>
<div class="note">Note: the predicted exchange rate change is always much smaller than the annualised interest differential, because it applies only to a single quarter (3 months = 1/4 of a year).</div>
</div>
<script>
function calcUIP(){
  var S0=1.1317;
  var iUS=parseFloat(document.getElementById("iUS").value);
  var iEU=parseFloat(document.getElementById("iEU").value);
  document.getElementById("vUS").textContent=iUS.toFixed(2);
  document.getElementById("vEU").textContent=iEU.toFixed(2);
  var S1=S0*(1+iUS/400)/(1+iEU/400);
  var diff=iUS-iEU; var diffQ=diff/4; var chg=(S1/S0-1)*100;
  document.getElementById("rDiff").textContent=diff.toFixed(2)+" pp";
  document.getElementById("rDiff").className="result-val"+(diff>0.05?" pos":diff<-0.05?" neg":" neu");
  document.getElementById("rDiffQ").textContent=diffQ.toFixed(3)+" pp";
  document.getElementById("rS1").textContent=S1.toFixed(4)+" USD/EUR";
  var sign=(chg>=0)?"+":"";
  document.getElementById("rChg").textContent=sign+chg.toFixed(3)+"%";
  document.getElementById("rChg").className="result-val"+(chg>0.001?" pos":chg<-0.001?" neg":" neu");
  var box=document.getElementById("interp");
  if(Math.abs(diff)<0.1){
    box.className="interp par";
    box.innerHTML="Interest rates are equal &mdash; UIP predicts <strong>no exchange rate change</strong> over the next 3 months.";
  } else if(diff>0){
    box.className="interp dep";
    box.innerHTML="US rates are higher by <strong>"+diff.toFixed(2)+" pp</strong> (annualised). Over the next <strong>3 months</strong>, UIP predicts a USD <strong>depreciation</strong> of "+Math.abs(chg).toFixed(3)+"% &mdash; just enough to offset the quarterly interest advantage of "+(diff/4).toFixed(3)+" pp.";
  } else {
    box.className="interp app";
    box.innerHTML="Euro Area rates are higher by <strong>"+Math.abs(diff).toFixed(2)+" pp</strong> (annualised). Over the next <strong>3 months</strong>, UIP predicts a USD <strong>appreciation</strong> of "+Math.abs(chg).toFixed(3)+"%.";
  }
}
calcUIP();
</script>
'
IRdisplay::display_html(html_code)

---

## 2. Theoretical Background

### The Uncovered Interest Rate Parity (UIP)

Imagine you have €1,000 and can choose between two investments:

**Option A – Invest in the Euro Area:**

$$\text{€}1{,}000 \cdot \left(1 + \frac{i_{EU}}{400}\right)$$

**Option B – Invest in the USA:**

Convert to USD at spot rate $S_t$, invest at the US rate, convert back after 3 months:

$$\text{€}1{,}000 \cdot S_t \cdot \left(1 + \frac{i_{US}}{400}\right) \cdot \frac{1}{S_{t+1}}$$

**The UIP condition** — both options yield the same return in equilibrium:

$$\boxed{S_{t+1}^{\text{UIP}} = S_t \cdot \frac{1 + i_{US}/400}{1 + i_{EU}/400}}$$

> **Note on the divisor 400:** FRED interest rates are annualised percentages. Dividing by 100 converts to decimals; dividing by 4 gives the **quarterly** rate — hence 400.

### Data Sources

| Variable | FRED Code | Description |
|---|---|---|
| $i_{US}$ | `DTB3` | 3-Month Treasury Bill Rate, USA |
| $i_{EU}$ | `IR3TIB01DEM156N` | 3-Month Interbank Rate, Germany (Euro Area proxy) |
| $S_t$ | `DEXUSEU` | USD per EUR, Spot Exchange Rate |

---

## 3. Interest Rate Developments: USA vs. Euro Area

The chart below shows the 3-month interest rates for the USA and the Euro Area from Q1 2022 to Q4 2025. Run the cell, then use the **interactive quiz** below to check your understanding.

In [ ]:
df |>
  select(qstart, i_US, i_EU) |>
  pivot_longer(c(i_US,i_EU), names_to="Region", values_to="Rate") |>
  mutate(Region=recode(Region,
    i_US="USA (3M Treasury Bill)",
    i_EU="Euro Area (3M Interbank DE)")) |>
ggplot(aes(x=qstart, y=Rate, color=Region, linetype=Region)) +
  geom_line(linewidth=1.3) + geom_point(size=3) +
  scale_color_manual(values=c(
    "USA (3M Treasury Bill)"="dodgerblue3",
    "Euro Area (3M Interbank DE)"="firebrick3")) +
  scale_x_date(date_breaks="6 months", date_labels="%b %Y") +
  labs(title="3-Month Interest Rates: USA vs. Euro Area (2022-2025)",
       subtitle="Beginning-of-quarter values | Source: FRED",
       x=NULL, y="Interest Rate (% p.a.)", color=NULL, linetype=NULL) +
  theme_minimal(base_size=13) +
  theme(legend.position="bottom",
        axis.text.x=element_text(angle=30,hjust=1),
        panel.grid.minor=element_blank())

In [ ]:
library(IRdisplay)
quiz_html <- '
<style>
.quiz-box{font-family:Calibri,Arial,sans-serif;max-width:700px;background:#f0f4ff;
  border:1px solid #b0c4de;border-radius:8px;padding:22px;margin:12px 0}
.quiz-title{font-size:16px;font-weight:bold;color:#1a3a6b;margin-bottom:16px}
.q-block{margin-bottom:18px;padding-bottom:14px;border-bottom:1px solid #ccd6f0}
.q-block:last-child{border-bottom:none;margin-bottom:0}
.q-text{font-size:14px;font-weight:bold;color:#212529;margin-bottom:8px}
.q-options label{display:block;margin:5px 0;font-size:14px;cursor:pointer;padding:4px 8px;border-radius:4px}
.q-options label:hover{background:#dce6ff}
.feedback{margin-top:8px;padding:8px 12px;border-radius:4px;font-size:13px;display:none}
.correct{background:#d4edda;color:#155724;border:1px solid #c3e6cb}
.wrong{background:#f8d7da;color:#721c24;border:1px solid #f5c6cb}
.btn-check{margin-top:10px;padding:7px 18px;background:#1f77b4;color:white;
  border:none;border-radius:4px;cursor:pointer;font-size:14px}
.btn-check:hover{background:#1560a0}
</style>
<div class="quiz-box">
<div class="quiz-title">&#128214; Task 1 — Quick Check: Interest Rate Developments</div>

<div class="q-block" id="q1">
<div class="q-text">a) In which period were US interest rates most significantly higher than Euro Area rates?</div>
<div class="q-options">
  <label><input type="radio" name="q1" value="a"> 2022 Q1 — both rates near zero</label>
  <label><input type="radio" name="q1" value="b"> 2022 Q3 to 2024 Q1 — US rates rose sharply while EU rates lagged</label>
  <label><input type="radio" name="q1" value="c"> 2025 Q3 — rates converged again</label>
</div>
<div class="feedback" id="f1a"></div>
<button class="btn-check" onclick="check(1)">Check answer</button>
</div>

<div class="q-block" id="q2">
<div class="q-text">b) According to UIP: if US rates are higher than Euro Area rates, what should happen to the USD?</div>
<div class="q-options">
  <label><input type="radio" name="q2" value="a"> USD should appreciate — investors prefer the higher yield</label>
  <label><input type="radio" name="q2" value="b"> USD should depreciate — the interest advantage must be offset by exchange rate loss</label>
  <label><input type="radio" name="q2" value="c"> USD should remain unchanged — UIP says rates determine only inflation</label>
</div>
<div class="feedback" id="f2a"></div>
<button class="btn-check" onclick="check(2)">Check answer</button>
</div>

<div class="q-block" id="q3">
<div class="q-text">c) When do US and Euro Area rates converge most noticeably in the chart?</div>
<div class="q-options">
  <label><input type="radio" name="q3" value="a"> Early 2022 — both start near zero</label>
  <label><input type="radio" name="q3" value="b"> Mid 2023 — Euro Area catches up as ECB raises rates</label>
  <label><input type="radio" name="q3" value="c"> Late 2024 / 2025 — both central banks begin cutting rates</label>
</div>
<div class="feedback" id="f3a"></div>
<button class="btn-check" onclick="check(3)">Check answer</button>
</div>

</div>
<script>
var answers = {1:"b", 2:"b", 3:"c"};
var explanations = {
  1: {
    b: "Correct! From around Q3 2022, the Fed raised rates aggressively while the ECB moved more slowly, creating a substantial differential peaking around 2023.",
    wrong: "Not quite. The largest differential emerged as the Fed tightened faster than the ECB, roughly Q3 2022 to Q1 2024."
  },
  2: {
    b: "Correct! UIP says the interest advantage must be exactly offset by exchange rate depreciation — otherwise arbitrage profits would exist.",
    wrong: "Not quite. UIP predicts the opposite: the higher-yielding currency must depreciate to eliminate arbitrage."
  },
  3: {
    c: "Correct! By late 2024 and into 2025, both the Fed and ECB began cutting rates, causing convergence.",
    wrong: "Look again at the chart. The most notable convergence happens toward the end of the sample as both central banks shifted to rate cuts."
  }
};
function check(qn){
  var sel = document.querySelector("input[name=q"+qn+"]:checked");
  var fb  = document.getElementById("f"+qn+"a");
  fb.style.display = "block";
  if(!sel){ fb.className="feedback wrong"; fb.textContent="Please select an answer first."; return; }
  if(sel.value === answers[qn]){
    fb.className="feedback correct";
    fb.textContent = "✓ " + explanations[qn][sel.value];
  } else {
    fb.className="feedback wrong";
    fb.textContent = "✗ " + explanations[qn]["wrong"];
  }
}
</script>
'
IRdisplay::display_html(quiz_html)

---

## 4. UIP Forecast vs. Actual Exchange Rate

The chart below compares the **UIP forecast** with the **actual realised exchange rate** at end-of-quarter (left axis, USD per EUR). The right axis shows the **percentage deviation** of the actual rate from the UIP forecast — a measure of how far off the prediction was each quarter.

In [ ]:
library(ggplot2)

p <- ggplot(df, aes(x=qstart)) +
  geom_col(aes(y=abweichung_pct/4 + mean(c(S_UIP,S_end)),
               fill="Deviation actual vs UIP (right axis)"),
           width=60, alpha=0.45, show.legend=TRUE) +
  geom_line(aes(y=S_UIP, color="UIP Forecast", linetype="UIP Forecast"), linewidth=1.2) +
  geom_point(aes(y=S_UIP, color="UIP Forecast"), shape=17, size=3.5) +
  geom_line(aes(y=S_end, color="Actual Rate", linetype="Actual Rate"), linewidth=1.2) +
  geom_point(aes(y=S_end, color="Actual Rate"), shape=16, size=3.5) +
  scale_color_manual(values=c("UIP Forecast"="#2ca02c","Actual Rate"="#d62728")) +
  scale_linetype_manual(values=c("UIP Forecast"="dashed","Actual Rate"="solid")) +
  scale_fill_manual(values=c("Deviation actual vs UIP (right axis)"="#9ecae1")) +
  scale_x_date(date_breaks="6 months", date_labels="%b %Y") +
  scale_y_continuous(
    name = "USD per EUR",
    sec.axis = sec_axis(
      trans = ~ (. - mean(c(df$S_UIP, df$S_end))) * 4,
      name  = "Deviation: Actual minus UIP Forecast (%)"
    )
  ) +
  labs(
    title    = "UIP Forecast vs. Actual USD/EUR Exchange Rate",
    subtitle = "Lines: exchange rates (left axis) | Bars: % deviation of actual from UIP forecast (right axis) | Source: FRED",
    x=NULL, color=NULL, linetype=NULL, fill=NULL
  ) +
  theme_minimal(base_size=13) +
  theme(legend.position="bottom",
        axis.text.x=element_text(angle=30,hjust=1),
        panel.grid.minor=element_blank(),
        axis.title.y.right=element_text(color="#4a90d9"),
        axis.text.y.right=element_text(color="#4a90d9"))
print(p)

**Reading this chart:** The green dashed line shows what the exchange rate *should have been* according to UIP; the red solid line shows what *actually happened*. The blue bars on the right axis measure the gap in percentage terms. Large bars indicate quarters where the UIP prediction was far off — the USD moved much more (or in the wrong direction) than the interest differential would predict. The systematic presence of large deviations is the core empirical challenge to the UIP.

> ### Task 2: UIP Forecast vs. Reality
>
> **a)** In which quarters does the actual exchange rate deviate most from the UIP forecast? Does the USD depreciate more or less than predicted?
>
> **b)** Are the deviations randomly distributed, or do you see systematic patterns?
>
> **c)** Overall, does the UIP appear to be a reliable short-run predictor of exchange rate movements?

**Your answers to Task 2:**

a) ...

b) ...

c) ...

---

## 5. Excess Volatility: Interest Differential vs. Exchange Rate Changes

A key feature of foreign exchange markets is **excess volatility**: actual exchange rate changes are far larger in magnitude than what the interest rate differential predicts. The chart below places both on a dual scale to make this contrast visible.

In [ ]:
max_fx  <- max(abs(df$delta_s_actual))
max_zdiff <- max(abs(df$zins_diff))
scale_f <- max_zdiff / max_fx

ggplot(df, aes(x=qstart)) +
  geom_col(aes(y=zins_diff, fill="Interest Differential (left axis)"),
           width=60, alpha=0.7) +
  geom_line(aes(y=delta_s_actual * scale_f,
                color="Actual FX Change (right axis)"), linewidth=1.3) +
  geom_point(aes(y=delta_s_actual * scale_f,
                 color="Actual FX Change (right axis)"), size=3) +
  geom_hline(yintercept=0, linewidth=0.6) +
  scale_fill_manual(values=c("Interest Differential (left axis)"="#aec7e8")) +
  scale_color_manual(values=c("Actual FX Change (right axis)"="#d62728")) +
  scale_x_date(date_breaks="6 months", date_labels="%b %Y") +
  scale_y_continuous(
    name = "Interest Rate Differential i_US - i_EU (pp, annualised)",
    sec.axis = sec_axis(trans=~./scale_f,
                        name="Actual USD/EUR Change over quarter (%)")
  ) +
  labs(
    title    = "Excess Volatility: Interest Differential vs. Actual Exchange Rate Change",
    subtitle = "Bars: interest differential at quarter start (left) | Line: quarterly FX change (right) | Source: FRED",
    x=NULL, fill=NULL, color=NULL
  ) +
  theme_minimal(base_size=13) +
  theme(legend.position="bottom",
        axis.text.x=element_text(angle=30,hjust=1),
        panel.grid.minor=element_blank(),
        axis.title.y.right=element_text(color="#d62728"),
        axis.text.y.right=element_text(color="#d62728"))

**Reading this chart:** The blue bars show how large the interest differential was at the start of each quarter — this is what drives the UIP forecast. The red line shows the actual exchange rate change that occurred. Notice that the red line swings far beyond what the bars would suggest: exchange rate fluctuations are typically **five to ten times larger** than the interest differential. This excess volatility means that factors other than interest rates — risk appetite, news shocks, global capital flows — dominate short-run exchange rate dynamics.

> ### Task 3: Excess Volatility
>
> **a)** Compare the magnitude of the interest differential (bars) to the actual exchange rate changes (line). How much larger are the FX fluctuations?
>
> **b)** Are the exchange rate changes at least consistent in *direction* with the interest differential?
>
> **c)** What does this suggest about the practical usefulness of UIP for short-run forecasting?

**Your answers to Task 3:**

a) ...

b) ...

c) ...

---

## 6. Statistical UIP Test: Regression Analysis

To formally test the UIP, economists estimate the following regression:

$$\Delta s_{t+1} = \alpha + \beta \cdot (i_{US,t} - i_{EU,t}) + \varepsilon_{t+1}$$

where $\Delta s_{t+1} = \left(\frac{S_{t+1}}{S_t} - 1\right) \times 100$ is the actual percentage change in the USD/EUR exchange rate over the quarter.

**Interpretation and empirical context:**  
(1) The UIP predicts $\alpha = 0$ and $\beta = 0.25$: a one percentage point higher annualised US rate implies a 0.25% quarterly USD depreciation.  
(2) If $\hat{\beta} \approx 0$, the interest differential has no predictive power for exchange rate changes — the UIP fails to hold in the data.  
(3) A **negative** $\hat{\beta}$ implies that high-interest currencies actually *appreciate* rather than depreciate, the exact opposite of UIP — this is the celebrated **Forward Premium Puzzle**, documented across many currency pairs and time periods.  
(4) The typically very low $R^2$ in such regressions confirms that short-run exchange rate dynamics are dominated by forces unrelated to interest differentials — risk premia, capital flow shocks, and market sentiment far outweigh the interest rate channel.

In [ ]:
reg <- lm(delta_s_actual ~ zins_diff, data=df)
summary(reg)

In [ ]:
beta_hat <- coef(reg)[2]
ggplot(df, aes(x=zins_diff, y=delta_s_actual)) +
  geom_point(size=3.5, color="#1f77b4", alpha=0.85) +
  geom_text(aes(label=paste0(year(qstart)," Q",quarter(qstart))),
            vjust=-0.9, size=3, color="#333333") +
  geom_smooth(method="lm", se=TRUE, color="#d62728", fill="#f7b6b6", linewidth=1.2) +
  geom_abline(intercept=0, slope=0.25,
              linetype="dashed", color="#2ca02c", linewidth=1.2) +
  annotate("text", x=Inf, y=Inf,
           label=paste0("OLS: beta = ", round(beta_hat,3)),
           hjust=1.1, vjust=1.8, size=4.5, color="#d62728") +
  annotate("text", x=Inf, y=-Inf,
           label="UIP prediction: beta = 0.25",
           hjust=1.1, vjust=-0.8, size=4.5, color="#2ca02c") +
  labs(title="UIP Test: Interest Rate Differential vs. Actual Exchange Rate Change",
       subtitle="Red: OLS estimate | Green dashed: UIP prediction (beta=0.25) | Source: FRED",
       x="Interest Rate Differential i_US - i_EU (pp, annualised)",
       y="Actual Exchange Rate Change (%, USD/EUR)") +
  theme_minimal(base_size=13) +
  theme(panel.grid.minor=element_blank())

> ### Task 4: Interpret the Regression Results
>
> **a)** What is the estimated $\hat{\beta}$? Is it significantly different from zero (p-value < 0.05)?
>
> **b)** The UIP predicts $\beta = 0.25$. How far does your estimate deviate? What does a positive or negative $\hat{\beta}$ imply?
>
> **c)** What does $R^2$ tell you about the explanatory power of the interest differential?
>
> **d)** Name two possible explanations for the empirical failure of UIP.

**Your answers to Task 4:**

a) ...

b) ...

c) ...

d) ...

---

## 7. Summary

| Concept | Key Message |
|---|---|
| **UIP Formula** | $S_{t+1}^{UIP} = S_t \cdot \frac{1+i_{US}/400}{1+i_{EU}/400}$ |
| **Interest Differential** | USA had significantly higher rates than the Euro Area in 2022–2023 |
| **UIP Forecast** | According to UIP, the USD should have depreciated substantially |
| **Reality** | Actual movements deviated considerably from UIP predictions |
| **Excess Volatility** | FX changes are far larger than the interest differential predicts |
| **Forward Premium Puzzle** | High-interest currencies tend to appreciate empirically — opposite of UIP |

---

### Task 5 — Final Reflection

In **3–5 sentences**: What do the results imply for an investor who tries to profit from interest differentials? What risks does the UIP framework overlook?

**Your answer to Task 5:**

...

---
*Data source: Federal Reserve Bank of St. Louis (FRED) — fred.stlouisfed.org*  
*Series: DTB3, IR3TIB01DEM156N, DEXUSEU*